# Camada Ouro - Agregações e Indicadores de Churn

Tabelas agregadas a partir da camada Silver pra análise de negócio e ML.

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

In [4]:
arquivo_silver = Path("../silver/telco_churn_silver.parquet")
pasta_gold = Path("../gold")
pasta_gold.mkdir(parents=True, exist_ok=True)
arquivo_sumario = pasta_gold / "churn_summary.parquet"
arquivo_matriz_risco = pasta_gold / "risk_matrix.parquet"
arquivo_importancia = pasta_gold / "feature_importance.parquet"
arquivo_ml = pasta_gold / "customer_features.parquet"

In [5]:
df_silver=pd.read_parquet(arquivo_silver)
resumo = df_silver.groupby(['tenure_group', 'Contract']).agg(
    taxa_churn=('Churn', 'mean'),
    total_clientes=('Churn', 'count'),
    media_cobranca_mensal=('MonthlyCharges', 'mean'),
    proporcao_addons=('has_internet_addons', 'mean')
).round(3)
resumo = resumo.reset_index()
print(resumo.head(10))

  tenure_group        Contract  taxa_churn  total_clientes  \
0        0-12m  Month-to-month       0.514            1994   
1        0-12m        One year       0.106             123   
2        0-12m        Two year       0.000              58   
3       13-24m  Month-to-month       0.377             737   
4       13-24m        One year       0.081             197   
5       13-24m        Two year       0.000              90   
6       25-48m  Month-to-month       0.329             802   
7       25-48m        One year       0.106             518   
8       25-48m        Two year       0.022             274   
9       49-72m  Month-to-month       0.260             342   

   media_cobranca_mensal  proporcao_addons  
0                 58.218             0.551  
1                 35.928             0.366  
2                 28.766             0.224  
3                 69.310             0.776  
4                 44.879             0.492  
5                 32.307             0.289  
6 

In [6]:
df_silver['faixa_cobranca'] = pd.cut(
    df_silver['MonthlyCharges'],
    bins=[0, 30, 60, 90, 120],
    labels=['Baixa (<$30)', 'Média ($30-60)', 'Alta ($60-90)', 'Premium (>$90)']
)
matriz_risco = df_silver.groupby(['tenure_group', 'faixa_cobranca'], observed=False).agg(
    taxa_churn=('Churn', 'mean'),
    clientes=('customerID', 'count')
).round(3)
matriz_risco = matriz_risco.reset_index()
print(matriz_risco)

   tenure_group  faixa_cobranca  taxa_churn  clientes
0         0-12m    Baixa (<$30)       0.227       600
1         0-12m  Média ($30-60)       0.433       527
2         0-12m   Alta ($60-90)       0.612       830
3         0-12m  Premium (>$90)       0.757       218
4        13-24m    Baixa (<$30)       0.047       255
5        13-24m  Média ($30-60)       0.229       214
6        13-24m   Alta ($60-90)       0.357       367
7        13-24m  Premium (>$90)       0.543       188
8        25-48m    Baixa (<$30)       0.023       351
9        25-48m  Média ($30-60)       0.119       285
10       25-48m   Alta ($60-90)       0.254       520
11       25-48m  Premium (>$90)       0.345       438
12       49-72m    Baixa (<$30)       0.014       441
13       49-72m  Média ($30-60)       0.072       237
14       49-72m   Alta ($60-90)       0.057       666
15       49-72m  Premium (>$90)       0.170       895


In [7]:
resumo.to_parquet(arquivo_sumario, index=False)
matriz_risco.to_parquet(arquivo_matriz_risco, index=False)
print(f"\nArquivos salvos com sucesso em: {pasta_gold}")
print(f"   - {arquivo_sumario.name} ({arquivo_sumario.stat().st_size / 1024:.1f} KB)")
print(f"   - {arquivo_matriz_risco.name} ({arquivo_matriz_risco.stat().st_size / 1024:.1f} KB)")


Arquivos salvos com sucesso em: ../gold
   - churn_summary.parquet (4.6 KB)
   - risk_matrix.parquet (3.4 KB)
